# teach-aws: Using & Evaluating the Final Model

Three things in this notebook:
1. **Use** our best model (Malay AWS QA, 87.3% strict accuracy on unseen questions)
2. **See the failure mode** (invented features) and the grounding guardrail that handles it
3. **Reproduce the evaluation** — same strict judge, three models:
   - our `teach-aws-qwen3.5-2b` (khursanirevo/teach-aws-qwen3.5-2b)
   - [`PixelSpaceAI/Malaysian-Qwen2.5-7B-AWS-Malay-LoRA`](https://huggingface.co/PixelSpaceAI/Malaysian-Qwen2.5-7B-AWS-Malay-LoRA)
   - its base, [`mesolitica/Malaysian-Qwen2.5-7B-Instruct`](https://huggingface.co/mesolitica/Malaysian-Qwen2.5-7B-Instruct)

Eval set: `PixelSpaceAI/aws-malay-qa` — 2,661 questions never trained on by any of the three.
Training code is private; everything here is inference-only.
New to LoRA finetuning? [This video](https://www.youtube.com/watch?v=zQi0kqQNDrU) covers the method behind the model.

In [ ]:
# 1) Setup (Colab GPU runtime, T4 works; A100/L4 faster)
%pip install -q vllm huggingface_hub

EVAL_ROWS = 200  # start smaller for Colab speed; set None for all 2,661

In [ ]:
# 2) Download the eval set (questions + reference answers)
from huggingface_hub import hf_hub_download
import json

rows = []
for fn in ['eval.jsonl', 'train.jsonl']:  # both splits are unseen test sets
    p = hf_hub_download('PixelSpaceAI/aws-malay-qa', fn, repo_type='dataset')
    rows += [json.loads(l) for l in open(p) if l.strip()]
print('eval rows:', len(rows))
if EVAL_ROWS:
    rows = rows[:EVAL_ROWS]

## Part A — Use our model

In [ ]:
from vllm import LLM, SamplingParams

OURS = 'khursanirevo/teach-aws-qwen3.5-2b'
llm = LLM(model=OURS, max_model_len=4096, gpu_memory_utilization=0.85,
           max_num_seqs=256, enforce_eager=True)
tok = llm.get_tokenizer()

def ask(questions):
    prompts = [tok.apply_chat_template([{'role': 'user', 'content': q}],
                                        tokenize=False, add_generation_prompt=True)
               for q in questions]
    outs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=2048))
    return [o.outputs[0].text.strip() for o in outs]

questions = [
    'Bolehkah S3 Transfer Acceleration melengkapi AWS Direct Connect?',
    'Bagaimanakah Step Functions menyokong keselarian?',
]
for q, a in zip(questions, ask(questions)):
    print('Q:', q, '\nA:', a[:400], '\n---')
del llm; import gc, torch; gc.collect(); torch.cuda.empty_cache()

## Part B — The failure mode and the guardrail

Real generation from our best model (judged FALSE):

> **Q:** Bolehkah saya menyimpan data saya apabila stack dipadamkan?
>
> **A:** "Ya, anda boleh menggunakan ciri **Memory and Cache Behavior** dalam AWS CloudFormation..." — feature does not exist. Correct answer: deletion policies with snapshots.

76% of the model's failures are fluent text like this. The guardrail flags answers that mention entities absent from the reference (in production: your retrieved passage).

In [ ]:
import re

ENT_RE = re.compile(
    r"\b(?:Amazon\s+[A-Z][a-zA-Z0-9]+|AWS\s+[A-Z][a-zA-Z0-9]+(?:\s+[A-Z][a-zA-Z0-9]+){0,2}"
    r"|[A-Z][a-zA-Z0-9]*(?:Flow|Front|Watch|Maker|Bridge|Formation|Sync|Lake|Base|Glue|Grid)"
    r"|[A-Z][a-zA-Z0-9]+\s(?:and\s)?[A-Z][a-zA-Z0-9]+(?:\s[A-Z][a-zA-Z0-9]+)*"
    r"|S3|EC2|EBS|Lambda|IAM|VPC|KMS|SSE|TLS|SSL|HTTP|DNS|SQL|API|SDK|CLI|MFA|HSM|WAF"
    r"|Route\s?53|Step\sFunctions|CloudFormation|CloudFront|CloudWatch|Direct\sConnect)\b"
)

def entities(text):
    return {m.group(0).lower().replace('amazon ', '').replace('aws ', '') for m in ENT_RE.finditer(text)}

def guard(answer, reference):
    novel = entities(answer) - entities(reference)
    if not novel:
        return answer
    return answer + '\n\n---\n⚠️ Unverified entities: ' + ', '.join(sorted(novel)) + '. Please verify with AWS docs.'

ref = 'CloudFormation membolehkan anda menentukan deletion policy untuk sumber dalam templat. Snapshot dicipta untuk volum Amazon EBS.'
hallucinated = 'Ya, anda boleh menggunakan ciri Memory and Cache Behavior dalam AWS CloudFormation untuk mengekalkan data.'
print(guard(hallucinated, ref))
print()
print('Benchmark on our full eval: catches 59% of wrong answers, false-flags 0.3% of correct ones.')

## Part C — Evaluate any model with the same strict judge

Judge: [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507), temperature 0, output structurally forced to TRUE/FALSE.
Rule: TRUE only if the candidate is **completely correct** — every reference fact present, nothing wrong or hallucinated. Partially correct = FALSE.

Models to compare (all unseen on the eval set):
- ours (`teach-aws-qwen3.5-2b`)
- `PixelSpaceAI/Malaysian-Qwen2.5-7B-AWS-Malay-LoRA` — a 7B register/style LoRA from the community
- `mesolitica/Malaysian-Qwen2.5-7B-Instruct` — its base

Note what this comparison measures: the PixelSpaceAI adapter was built for voice/register
(its card recommends RAG for factual accuracy), so under a fact-strict judge we expect it to
shine on style and struggle on completeness — which is exactly the interesting datapoint.

In [ ]:
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams
from huggingface_hub import snapshot_download

JUDGE = 'Qwen/Qwen3-4B-Instruct-2507'
JUDGE_SYSTEM = (
    'You are a strict grader. Given a question, a REFERENCE answer, and a CANDIDATE answer, '
    'output exactly one word: TRUE or FALSE.\n'
    'TRUE only if the candidate is COMPLETELY correct: every factual element of the reference '
    'is present and correct, and nothing wrong or hallucinated is added. Partially correct = '
    'FALSE. Ignore style/wording/formatting; judge factual content only.'
)

def generate_answers(model_id, is_lora_adapter=False):
    """Load model (merging LoRA adapters first if needed), answer all eval questions."""
    if is_lora_adapter:
        from peft import PeftModel
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch
        base = AutoModelForCausalLM.from_pretrained(
            'mesolitica/Malaysian-Qwen2.5-7B-Instruct', dtype=torch.bfloat16, device_map='cuda')
        m = PeftModel.from_pretrained(base, model_id)
        m = m.merge_and_unload()
        path = snapshot_download(model_id).replace('/snapshots/' + model_id.split('/')[-1], '')
        out_dir = '/tmp/merged_' + model_id.split('/')[-1]
        m.save_pretrained(out_dir)
        AutoTokenizer.from_pretrained(model_id).save_pretrained(out_dir)
        del m, base; gc.collect(); torch.cuda.empty_cache()
        model_id = out_dir
    llm = LLM(model=model_id, max_model_len=4096, gpu_memory_utilization=0.85,
              max_num_seqs=256, enforce_eager=True)
    t = llm.get_tokenizer()
    prompts = [t.apply_chat_template([{'role': 'user', 'content': r['messages'][0]['content']}],
                                      tokenize=False, add_generation_prompt=True) for r in rows]
    outs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=2048))
    answers = [o.outputs[0].text.strip() for o in outs]
    del llm; gc.collect(); torch.cuda.empty_cache()
    return answers

def judge_answers(answers):
    jl = LLM(model=JUDGE, max_model_len=8192, gpu_memory_utilization=0.85,
             max_num_seqs=256, enforce_eager=True)
    jt = jl.get_tokenizer()
    jprompts = [jt.apply_chat_template(
        [{'role': 'system', 'content': JUDGE_SYSTEM},
         {'role': 'user', 'content': f"Question: {r['messages'][0]['content']}\n\nREFERENCE answer:\n{r['messages'][1]['content']}\n\nCANDIDATE answer:\n{a}"}],
        tokenize=False, add_generation_prompt=True) for r, a in zip(rows, answers)]
    jouts = jl.generate(jprompts, SamplingParams(
        temperature=0.0, max_tokens=8,
        structured_outputs=StructuredOutputsParams(choice=['TRUE', 'FALSE'])))
    verdicts = [o.outputs[0].text.strip().upper().startswith('TRUE') for o in jouts]
    del jl; gc.collect(); torch.cuda.empty_cache()
    return verdicts

In [ ]:
# 3) Run the comparison
import gc, torch

results = {}
for name, model_id, is_lora in [
    ('ours (teach-aws 2B)', OURS, False),
    ('PixelSpaceAI 7B LoRA', 'PixelSpaceAI/Malaysian-Qwen2.5-7B-AWS-Malay-LoRA', True),
    ('mesolitica base 7B', 'mesolitica/Malaysian-Qwen2.5-7B-Instruct', False),
]:
    print(f'=== {name} ===')
    answers = generate_answers(model_id, is_lora)
    verdicts = judge_answers(answers)
    acc = sum(verdicts) / len(verdicts)
    results[name] = {'acc': acc, 'answers': answers, 'verdicts': verdicts}
    print(f'{name}: {acc:.1%} ({sum(verdicts)}/{len(verdicts)})\n')

print('=== SUMMARY (strict judge, n=%d) ===' % len(rows))
for name, r in sorted(results.items(), key=lambda kv: -kv[1]['acc']):
    print(f"{r['acc']:.1%}  {name}")

## Reference numbers (our full-run results, n=2,661)

| Model | Strict acc |
|---|---|
| teach-aws Qwen3.5-2B (ours) | **87.3%** |
| base Qwen3.5-2B | 4.1% |

Your Colab numbers will differ slightly at small n (±7pp at n=200, ±1.7pp at n=343).
Set `EVAL_ROWS = None` to reproduce the full-set numbers.

## Reading the comparison fairly

- **Style vs facts:** the PixelSpaceAI adapter was trained (~2.5k pairs) for voice/register and
  recommends RAG for accuracy; ours was trained for fact recall on this corpus. Under a
  fact-strict judge the gap measures *task fit*, not model quality.
- **Look at the generations, not just the score:** flip through `results[name]['answers']` —
  you'll see register differences (markdown vs prose) the judge deliberately ignores.
- **The judge is a model too.** Small-model strict-binary verdicts have their own error rate;
  spot-check ~10 FALSEs before quoting numbers.